In [9]:
#!/usr/bin/env python3
"""
Генерация таблицы свойств воды и водяного пара по IAPWS-IF97.

Столбцы:
  p  — давление, МПа
  t  — температура, °C
  h  — удельная энтальпия, кДж/кг
  s  — удельная энтропия, кДж/(кг·К)
  y  — влажность:
        -1       — однофазная среда (вода / перегретый пар)
         0       — сухой насыщенный пар  (x = 1)
         1       — кипящая жидкость      (x = 0)
         0<y<1   — влажный пар           (y = 1 − x)
  v  — удельный объём, м³/кг

Установка:  pip install iapws numpy pandas
"""

import sys
import time
import numpy as np
import pandas as pd
from iapws import IAPWS97

# ================================================================
#  НАСТРОЙКИ  (редактируйте под свою задачу)
# ================================================================
CONFIG = {
    # --- Сетка давлений: (начало, конец, шаг) в МПа ---
    "p_ranges": [
        (0.002,  0.010,  0.0005),    #   2 …  10 кПа
        (0.010,  0.100,  0.0010),    #  10 … 100 кПа
        (0.100,  1.000,  0.0100),    # 0.1 …   1 МПа
        (1.000,  5.000,  0.0500),    #   1 …   5 МПа
        (5.000, 10.000,  0.1000),    #   5 …  10 МПа
        (10.00, 22.000,  0.1000),    #  10 …  22 МПа
        (23.00, 30.000,  0.1000),    #  23 …  30 МПа  (сверхкритика)
    ],

    # --- Температура, °C ---
    "t_max":             600.0,     # верхний предел
    "dt_superheat":       1.0,     # шаг — перегретый пар
    "dt_subcooled":       1.0,     # шаг — подохлаждённая вода
    "t_min_subcooled":    5.0,     # нижний предел подохл. воды
    "dt_supercritical":   1.0,     # шаг — сверхкритическая область
    "t_min_supercrit":   100.0,     # нижний предел сверхкритики

    # Доп. отступы от Tнас для точек вблизи линии насыщения, °C
    "near_sat_offsets": [0.5, 2.0, 5.0],

    # --- Влажность (двухфазная область) ---
    "y_step": 0.01,                 # шаг по y = 1 − x

    # --- Опции ---
    "include_subcooled": True,
    "output": "WATER_STEAM_PROP",
}

P_CRIT = 22.064  # критическое давление воды, МПа


# ================================================================
#  ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ================================================================

def _build_pressures(p_ranges):
    """Собрать отсортированный массив уникальных давлений."""
    parts = []
    for a, b, step in p_ranges:
        parts.append(np.arange(a, b + step * 0.01, step))
    return np.unique(np.round(np.concatenate(parts), 8))


def _pt(**kw):
    """Безопасный вызов IAPWS97.  Возвращает объект или None."""
    try:
        obj = IAPWS97(**kw)
        if obj.v is not None and obj.v > 0:
            return obj
    except Exception:
        pass
    return None


def _unique_sorted(temps):
    """Убрать дубли и отсортировать массив температур."""
    return sorted(set(round(t, 4) for t in temps))


# ================================================================
#  ОСНОВНАЯ ФУНКЦИЯ ГЕНЕРАЦИИ
# ================================================================

def generate(cfg=None):
    """
    Построить DataFrame со свойствами воды / пара.

    Parameters
    ----------
    cfg : dict | None
        Настройки (по умолчанию — CONFIG).

    Returns
    -------
    pd.DataFrame  (столбцы: p, t, h, s, y, v)
    """
    if cfg is None:
        cfg = CONFIG

    pressures   = _build_pressures(cfg["p_ranges"])
    y_step      = cfg["y_step"]
    y_vals      = np.round(np.arange(0.0, 1.0 + y_step * 0.1, y_step), 6)
    t_max       = cfg["t_max"]
    dt_sh       = cfg["dt_superheat"]
    dt_sc       = cfg["dt_subcooled"]
    dt_spc      = cfg["dt_supercritical"]
    t_min_sub   = cfg["t_min_subcooled"]
    t_min_spc   = cfg["t_min_supercrit"]
    offsets     = cfg.get("near_sat_offsets", [])
    do_subcool  = cfg.get("include_subcooled", True)

    rows = []
    n_p  = len(pressures)

    for ip, p in enumerate(pressures, 1):
        sys.stdout.write(f"\r  [{ip}/{n_p}]  p = {p:.4f} МПа")
        sys.stdout.flush()

        # ========== Докритическое давление ==========
        if p < P_CRIT:
            sat = _pt(P=p, x=0)
            if sat is None:
                continue
            t_sat = round(sat.T - 273.15, 4)

            # 1) Двухфазная область + линия насыщения
            for y_val in y_vals:
                x = round(1.0 - y_val, 6)
                obj = _pt(P=p, x=x)
                if obj:
                    rows.append([p, t_sat, obj.h, obj.s,
                                 round(y_val, 4), obj.v])

            # 2) Подохлаждённая вода
            if do_subcool and t_sat > t_min_sub + 1:
                temps = list(np.arange(t_min_sub, t_sat, dt_sc))
                for off in offsets:
                    tn = t_sat - off
                    if tn > t_min_sub:
                        temps.append(tn)
                for t in _unique_sorted(temps):
                    obj = _pt(T=t + 273.15, P=p)
                    if obj:
                        rows.append([p, round(t, 4), obj.h, obj.s,
                                     -1.0, obj.v])

            # 3) Перегретый пар
            temps = list(np.arange(t_sat + dt_sh, t_max + dt_sh * 0.01,
                                   dt_sh))
            for off in offsets:
                tn = t_sat + off
                if tn < t_max:
                    temps.append(tn)
            for t in _unique_sorted(temps):
                if t > t_max:
                    break
                obj = _pt(T=t + 273.15, P=p)
                if obj:
                    rows.append([p, round(t, 4), obj.h, obj.s,
                                 -1.0, obj.v])

        # ========== Сверхкритическое давление ==========
        else:
            temps = np.arange(t_min_spc, t_max + dt_spc * 0.01, dt_spc)
            for t in temps:
                obj = _pt(T=t + 273.15, P=p)
                if obj:
                    rows.append([p, round(t, 4), obj.h, obj.s,
                                 -1.0, obj.v])

    print()  # перенос строки

    df = pd.DataFrame(rows, columns=["p", "t", "h", "s", "y", "v"])
    df.sort_values(["p", "y", "t"], ascending=[True, False, True],
                   inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


# ================================================================
#  ЗАПУСК
# ================================================================

if __name__ == "__main__":
    print("═" * 58)
    print("  Таблица свойств воды / пара  ·  IAPWS-IF97")
    print("═" * 58)

    t0 = time.time()
    df = generate()
    elapsed = time.time() - t0

    # Сохранение
    out = CONFIG["output"]
    for param in ['h', 's', 'y', 'v']:
        temp = df[['p', 't', param]]
        temp.to_excel(f'{param}_{out}.xlsx', index=False, float_format="%.8g")
        if param == 'h' or param == 'y':
            temp = df[['p', 's', param]]
            temp.to_excel(f'p_s_{param}_{out}.xlsx', index=False, float_format="%.8g")
            

        
    #df.to_excel(out, index=False, float_format="%.8g")


    # Отчёт
    n1 = int((df["y"] == -1).sum())
    n2 = int((df["y"] != -1).sum())
    print(f"\n  Всего точек : {len(df):>6}   ({elapsed:.1f} с)")
    print(f"  Однофазных  : {n1:>6}")
    print(f"  Двухфазных  : {n2:>6}")
    print(f"  p           : {df['p'].min():.4f} … {df['p'].max():.1f} МПа")
    print(f"  t           : {df['t'].min():.1f} … {df['t'].max():.1f} °C")
    print(f"\n  Файл → {out}\n")
    print(df.head(20).to_string(index=False))

══════════════════════════════════════════════════════════
  Таблица свойств воды / пара  ·  IAPWS-IF97
══════════════════════════════════════════════════════════
  [518/518]  p = 30.0000 МПа

  Всего точек : 348471   (55.0 с)
  Однофазных  : 303324
  Двухфазных  :  45147
  p           : 0.0020 … 30.0 МПа
  t           : 5.0 … 600.0 °C

  Файл → WATER_STEAM_PROP

    p       t          h        s    y         v
0.002 17.4953  73.434592 0.260583 1.00  0.001001
0.002 17.4953  98.029362 0.345204 0.99  0.670888
0.002 17.4953 122.624131 0.429826 0.98  1.340774
0.002 17.4953 147.218901 0.514447 0.97  2.010660
0.002 17.4953 171.813671 0.599069 0.96  2.680547
0.002 17.4953 196.408441 0.683690 0.95  3.350433
0.002 17.4953 221.003210 0.768311 0.94  4.020319
0.002 17.4953 245.597980 0.852933 0.93  4.690206
0.002 17.4953 270.192750 0.937554 0.92  5.360092
0.002 17.4953 294.787519 1.022176 0.91  6.029978
0.002 17.4953 319.382289 1.106797 0.90  6.699865
0.002 17.4953 343.977059 1.191418 0.89  7.3697

In [8]:
IAPWS97(P=10, T=273.15+300).h

np.float64(1343.0966090616594)